In [1]:
from pathlib import Path
from data_wrangler import DataWrangler, LocalCsvDataSource

In [2]:
# Configure logging
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


In [3]:
DOWNLOAD_PATH = Path("../../data/land_registry_data/")
assert DOWNLOAD_PATH.exists(), f"Download path {DOWNLOAD_PATH} does not exist. Please download the data and place it in this directory."

In [4]:
ONELAKE_PATH = Path("../../data/onelake")
assert ONELAKE_PATH.exists(), f"OneLake path {ONELAKE_PATH} does not exist. Please set up OneLake and place the data in this directory."

In [5]:
data_source = LocalCsvDataSource(
    data_folder=str(DOWNLOAD_PATH),
    column_names=DataWrangler.COLUMN_NAMES,
    output_root=str(ONELAKE_PATH)
)
house_price_summary = DataWrangler.run_pipeline_with_data_source(data_source)

2026-04-24 11:24:34,824 - INFO - Starting process_to_silver pipeline
2026-04-24 11:24:34,825 - INFO - Scanning price paid data from source...
2026-04-24 11:24:34,827 - INFO -   → Dropping records without postcode
2026-04-24 11:24:34,828 - INFO -   → Dropping records without date
2026-04-24 11:24:34,829 - INFO -   → Filtering 'Other' property types
2026-04-24 11:24:34,829 - INFO -   → Extracting year from date
2026-04-24 11:24:34,829 - INFO -   → Renaming property type values
2026-04-24 11:24:34,830 - INFO -   → Renaming duration values
2026-04-24 11:24:34,831 - INFO -   → Renaming old/new values
2026-04-24 11:24:34,832 - INFO -   → Extracting postcode area
2026-04-24 11:24:34,833 - INFO - Collecting results...
2026-04-24 11:24:36,505 - INFO - Collected 9,417,640 records
2026-04-24 11:24:36,506 - INFO - Writing to silver.price_paid_data (mode=overwrite)...
2026-04-24 11:24:45,385 - INFO - process_to_silver complete
2026-04-24 11:24:45,400 - INFO - Starting project_to_gold pipeline
2026-

## Consume Data From Gold Layer

In [6]:
import polars as pl

In [7]:
prices = pl.read_delta(ONELAKE_PATH / "gold" / "fact_price_paid")
dates = pl.read_delta(ONELAKE_PATH / "gold" / "dim_date")

In [8]:
house_price_summary = (
    prices
    .join(dates, left_on="date_of_transfer", right_on="date")
    .group_by(["year", "property_type"])
    .agg(pl.median("price").alias("median_price")).sort(["year", "property_type"])
)

In [9]:
import plotly.express as px
fig = px.line(
    house_price_summary,
    x="year",
    y="median_price",
    color="property_type",
    markers=True,
    title="Median House Price by Year and Property Type",
    labels={"year": "Year", "median_price": "Median Price (£)", "property_type": "Property Type"},
)
fig.show()